# Week 7 · Day 3 — Build & Ship: The Gapminder Explorer (Live Web App)

Today is the big one. You will build a complete interactive data-story app with
**two widgets**, run it, push it to GitHub, and **deploy it live** so anyone in
the world can open it with a link.

**What we build:** *Gapminder Explorer* — pick a continent from a dropdown and a
year with a slider, and the app updates its table and charts.

**Plan for the class:**
1. Build the app piece by piece (in this notebook we write it to `app.py`).
2. Run it locally to test.
3. Push everything to GitHub.
4. Deploy to Streamlit Community Cloud → get a public link.

Let's go!


## 1. Project folder

A Streamlit project needs, at minimum:

```
gapminder-explorer/
├── app.py            <- the app
├── gapminder.csv     <- the data
└── requirements.txt  <- the libraries the app needs
```

We build each of these files now. First, make sure your `gapminder.csv` is in
the same folder as the notebook.

In [ ]:
import os
# Confirm the data file is reachable. Copy it next to the notebook if needed.
import shutil
src = "../Day1/data/gapminder.csv"
if os.path.exists(src) and not os.path.exists("gapminder.csv"):
    shutil.copy(src, "gapminder.csv")
print("gapminder.csv present:", os.path.exists("gapminder.csv"))

## 2. Build the app step by step

We will write the whole app into a variable, then save it to `app.py`.
Read every section — it is all Python you already know, plus `st.` commands.

### Section by section, what the app does:

- **Title and intro** — a heading and a sentence of story
- **Sidebar widgets** — a `st.selectbox` for continent, a `st.slider` for year
- **Filtered table** — show the data for the chosen continent and year
- **Three charts** — a trend line, a comparison bar, a relationship scatter
- **Takeaway** — a closing sentence


In [ ]:
app_code = r'''
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# ---------- Load data ----------
df = pd.read_csv("gapminder.csv")

# ---------- Title and intro ----------
st.title("Gapminder Explorer")
st.write(
    "Explore how life expectancy, income, and population changed around the "
    "world from 1952 to 2007. Use the controls on the left to filter the data."
)

# ---------- Sidebar widgets ----------
st.sidebar.header("Controls")

# Widget 1: a dropdown to pick a continent
continents = sorted(df["continent"].unique())
chosen_continent = st.sidebar.selectbox("Choose a continent", continents)

# Widget 2: a slider to pick a year
years = sorted(df["year"].unique())
chosen_year = st.sidebar.slider(
    "Choose a year",
    min_value=int(min(years)),
    max_value=int(max(years)),
    value=2007,
    step=5,
)

# ---------- Filter the data ----------
filtered = df[(df["continent"] == chosen_continent) & (df["year"] == chosen_year)]

st.header(f"{chosen_continent} in {chosen_year}")
st.write(f"Showing {len(filtered)} countries.")
st.dataframe(filtered)

# ---------- Chart 1: trend over time for the chosen continent ----------
st.subheader("Average life expectancy over time")
trend = df[df["continent"] == chosen_continent].groupby("year")["lifeExp"].mean()

fig1, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(trend.index, trend.values, marker="o", color="teal")
ax1.set_title(f"Average Life Expectancy in {chosen_continent} (1952-2007)")
ax1.set_xlabel("Year")
ax1.set_ylabel("Life Expectancy (years)")
st.pyplot(fig1)

# ---------- Chart 2: comparison bar for the chosen year ----------
st.subheader("Continents compared in the chosen year")
year_data = df[df["year"] == chosen_year]

fig2, ax2 = plt.subplots(figsize=(8, 4))
sns.barplot(data=year_data, x="continent", y="lifeExp", ax=ax2)
ax2.set_title(f"Average Life Expectancy by Continent ({chosen_year})")
ax2.set_xlabel("Continent")
ax2.set_ylabel("Life Expectancy (years)")
st.pyplot(fig2)

# ---------- Chart 3: relationship scatter for the chosen year ----------
st.subheader("Income vs life expectancy in the chosen year")
fig3, ax3 = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=year_data, x="gdpPercap", y="lifeExp",
                hue="continent", alpha=0.7, ax=ax3)
ax3.set_title(f"Income vs Life Expectancy ({chosen_year})")
ax3.set_xlabel("GDP per person")
ax3.set_ylabel("Life Expectancy (years)")
st.pyplot(fig3)

# ---------- Takeaway ----------
st.header("Takeaway")
st.write(
    "Life expectancy rose almost everywhere over these decades, but a clear gap "
    "between continents remains. Income and life expectancy move together, though "
    "the link is strongest among lower-income countries."
)
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("Saved app.py  (", len(app_code), "characters )")

## 3. The requirements file

Streamlit Community Cloud needs to know which libraries to install.
We list them in a file named `requirements.txt`.

In [ ]:
requirements = """streamlit
pandas
matplotlib
seaborn
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("Saved requirements.txt")
print(requirements)

## 4. Run the app locally to test it

**This is the safe fallback — always test locally before deploying.**

Streamlit apps do **not** run inside a notebook. Open a **terminal** in the same
folder as `app.py` and type:

```
streamlit run app.py
```

A browser tab opens at `http://localhost:8501` showing your app. Try the
dropdown and the slider — the charts update as you change them.

**In Google Colab** (where there is no local browser), test the app like this
instead, in a Colab cell:

```
!pip install streamlit -q
!npm install -g localtunnel
!streamlit run app.py &>/dev/null &
!npx localtunnel --port 8501
```

Click the link localtunnel prints. If it works locally, it will work when
deployed. **If anything breaks, fix it here first — do not deploy a broken app.**


## 5. Push to GitHub

Deployment reads your app **from GitHub**, so the files must be in a repo.

1. Create a new repo on GitHub named `gapminder-explorer` (public).
2. Put these three files in it: `app.py`, `gapminder.csv`, `requirements.txt`.
3. Add a `README.md` (next section).

If you use the terminal:

```
git init
git add app.py gapminder.csv requirements.txt README.md
git commit -m "Gapminder Explorer Streamlit app"
git branch -M main
git remote add origin https://github.com/YOUR-USERNAME/gapminder-explorer.git
git push -u origin main
```

Or drag-and-drop the files using the **Add file → Upload files** button on the
GitHub website — that works perfectly too.


## 6. A README that sells the project

Every portfolio project needs a README. We write one to a file.
Replace `YOUR-USERNAME` and, after deploying, paste your live link.

In [ ]:
readme = """# Gapminder Explorer

An interactive data-story web app built with Streamlit. Explore how life
expectancy, income, and population changed across the world from 1952 to 2007.

## Live app
[Open the live app](https://YOUR-APP-LINK.streamlit.app)  <!-- paste your link here -->

## What it does
- Pick a **continent** (dropdown) and a **year** (slider)
- See a filtered data table
- View three charts: a life-expectancy trend, a continent comparison, and an
  income-vs-life-expectancy relationship

## Built with
Python, pandas, Matplotlib, Seaborn, Streamlit

## Data
Gapminder dataset (life expectancy, population, GDP per capita by country/year).

## Run locally
```
pip install -r requirements.txt
streamlit run app.py
```
"""

with open("README.md", "w") as f:
    f.write(readme)

print("Saved README.md")

## 7. Deploy to Streamlit Community Cloud

Now the exciting part — put it on the internet, free.

1. Go to **share.streamlit.io** and sign in with your **GitHub** account.
2. Click **Create app** → **Deploy a public app from GitHub**.
3. Choose your `gapminder-explorer` repo, branch `main`, main file `app.py`.
4. Click **Deploy**. Wait 1-3 minutes while it installs and boots.
5. You get a public link like `https://your-name-gapminder.streamlit.app`.

**Share that link in the WhatsApp group** — it is a live app on your portfolio!

### If deploy shows an error
- **ModuleNotFoundError** → a library is missing from `requirements.txt`. Add it, push, and the app redeploys automatically.
- **FileNotFoundError: gapminder.csv** → the CSV is not in the repo, or the name/case does not match. Make sure `gapminder.csv` is in the same folder as `app.py`.
- **App is asleep** → free apps sleep after inactivity; just click "wake up".

Because you tested locally in step 4, deploy problems are usually just a missing
file or library — quick to fix.


## 8. Recap — you shipped a live app!

Today you:

- Built a full **Streamlit app** with **two widgets** (dropdown + slider)
- Showed a filtered table and three charts that respond to the widgets
- Wrote `requirements.txt` and a portfolio **README**
- **Tested locally** as a safety net
- Pushed to **GitHub** and **deployed live** to Streamlit Community Cloud

This live link is one of the strongest pieces in your portfolio — a recruiter can
click it and use your work in seconds.

Your assignment: make this app **your own**. See the assignment notebook.
